# 14 Experimental Design and Replicates - Python

## Biochemistry question

How do controls, replicate counts, and variability affect the way we interpret a synthetic enzyme activity experiment?

This notebook uses synthetic data to practice experimental design thinking. It is not a real enzyme study and should not be used for biological, clinical, diagnostic, or regulatory conclusions.


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"


## 1. Setup

We will use `pandas` for tables and `plotly` for visual checks. The dataset has three groups: control, low inhibitor, and high inhibitor.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DATA_PATH = Path("../data/enzyme_activity/enzyme_activity_three_groups.csv")


## 2. Dataset Preview

Before interpreting any result, first check the rows, columns, groups, and replicate structure.


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.groupby("group").agg(
    n=("enzyme_activity", "count"),
    mean_activity=("enzyme_activity", "mean"),
    sd_activity=("enzyme_activity", "std"),
).reset_index()


## 3. What Counts as a Replicate?

In this teaching dataset, each row is treated as one replicate measurement. In real laboratory work, it is important to know whether replicates are technical replicates, biological replicates, or repeated measurements from the same source.


In [ ]:
replicate_counts = df.groupby("group").size().reset_index(name="replicate_count")
replicate_counts


## 4. Visual Check of Individual Measurements

A plot of individual points helps you see variation inside each group. This is often more informative than looking only at group means.


In [ ]:
fig = px.strip(
    df,
    x="group",
    y="enzyme_activity",
    color="group",
    hover_data=["sample_id"],
    title="Synthetic Enzyme Activity: Individual Replicates",
)
fig.update_yaxes(title="Enzyme Activity")
# If this chart does not render in Jupyter, try: fig.show(renderer="iframe") or fig.show(renderer="browser")
fig.show()


## 5. Summary with SEM

SEM helps describe uncertainty around the estimated group mean. It is not the same thing as the spread of individual observations.


In [ ]:
summary = (
    df.groupby("group")
    .agg(
        mean_activity=("enzyme_activity", "mean"),
        sd_activity=("enzyme_activity", "std"),
        n=("enzyme_activity", "count"),
    )
    .reset_index()
)
summary["sem_activity"] = summary["sd_activity"] / (summary["n"] ** 0.5)
summary


In [ ]:
fig = px.bar(
    summary,
    x="group",
    y="mean_activity",
    error_y="sem_activity",
    title="Mean Enzyme Activity by Experimental Group",
    labels={"mean_activity": "Mean Enzyme Activity"},
)
# If this chart does not render in Jupyter, try: fig.show(renderer="iframe") or fig.show(renderer="browser")
fig.show()


## 6. Design Review Table

This table turns the analysis into experimental-design questions. The goal is to practice asking what would make an interpretation stronger.


In [ ]:
design_review = pd.DataFrame({
    "design_question": [
        "Is there a control group?",
        "Are there multiple treatment levels?",
        "Are replicate counts balanced?",
        "Is variability visible within groups?",
        "Can this dataset prove mechanism?",
    ],
    "student_note": [
        "Yes, the control group provides a baseline for comparison.",
        "Yes, low and high inhibitor groups allow a dose-like comparison.",
        "Yes, each group has the same number of synthetic replicates.",
        "Yes, individual replicate plots show spread within each group.",
        "No, this dataset only supports cautious pattern description.",
    ],
})
design_review


## What You Should Notice

- The control group has the highest mean enzyme activity in this synthetic dataset.
- The low inhibitor group is lower than control, and the high inhibitor group is lower still.
- Replicate counts are balanced, which makes group summaries easier to compare.
- Individual replicate plots help show whether the mean hides important spread.

## Interpretation Practice

- How would you describe the difference between the three groups in one cautious sentence?
- Why is it useful to show individual points as well as means?
- What is the difference between SD and SEM in this notebook?
- What extra information would you want before treating this as a real experiment?

## Common Mistake

- Do not say the inhibitor "proves" a mechanism. The synthetic data show a pattern that is consistent with lower enzyme activity in the inhibitor groups.

## Limitations

- This is synthetic learning data.
- The notebook does not distinguish technical and biological replicates from real lab records.
- The analysis does not test mechanism, specificity, or real-world biological meaning.
- More experimental detail would be needed for a real study.
